# Telecom Customer Churn & Retention
## 04 — Feature Engineering

This notebook prepares the churn dataset for machine-learning models.

The feature-engineering process is designed to be:

- **leakage-safe** — outcome-derived fields are excluded;
- **reproducible** — train/test splitting and preprocessing are deterministic;
- **interpretable** — engineered variables have clear business meaning;
- **model-ready** — categorical and numeric features are prepared through a reusable scikit-learn preprocessing pipeline.

### Note

`CLTV` is **not used as a churn predictor** in this project.

Although CLTV is valuable for the business, the project will use it **after churn prediction** to prioritize high-value customers for retention. Keeping churn risk and customer value separate makes the final retention framework easier to interpret:

**Predicted churn risk + CLTV → retention priority**

The prediction target is:

- `Churn Value = 1` → churned
- `Churn Value = 0` → retained


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the modeling dataset


In [2]:
candidate_paths = [
    Path("../data/processed/telco_churn_modeling.csv"),
    Path("data/processed/telco_churn_modeling.csv"),
    Path("telco_churn_modeling.csv"),
    Path("/mnt/data/processed/telco_churn_modeling.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "telco_churn_modeling.csv was not found. "
        "Run 01_data_preparation.ipynb first."
    )

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Loaded: ../data/processed/telco_churn_modeling.csv
Rows: 7,043
Columns: 21


In [3]:
df.head()


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,"3,046.05",1,5003
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,"5,036.30",1,5340


## 2. Confirm target and modeling fields


In [4]:
TARGET = "Churn Value"

if TARGET not in df.columns:
    raise KeyError(f"{TARGET} is missing from the dataset.")

print("Target distribution:")
display(
    df[TARGET]
    .value_counts()
    .rename(index={0: "Retained", 1: "Churned"})
    .to_frame("customers")
)

print(f"Overall churn rate: {100 * df[TARGET].mean():.2f}%")


Target distribution:


,customers
Churn Value,
Retained,5174
Churned,1869


Overall churn rate: 26.54%


## 3. Separate CLTV from predictive features

`CLTV` will be retained as a business-value variable but excluded from the churn model.

It will later be combined with predicted churn probabilities to classify customers into retention-priority groups.


In [5]:
customer_value = df[["CLTV", TARGET]].copy()

model_df = df.drop(columns=["CLTV"]).copy()

print(f"Columns before CLTV exclusion: {df.shape[1]}")
print(f"Columns available for modeling: {model_df.shape[1]}")


Columns before CLTV exclusion: 21
Columns available for modeling: 20


## 4. Engineer business-relevant features

The following features are created:

### `Num Addon Services`
Counts how many internet add-on services the customer currently uses:

- Online Security
- Online Backup
- Device Protection
- Tech Support
- Streaming TV
- Streaming Movies

### `Automatic Payment`
Indicates whether the customer uses an automatic bank-transfer or credit-card payment method.

### `Month-to-Month`
Flags customers on month-to-month contracts.

These features are intended to make important business patterns easier for models to capture while preserving the original variables.


In [6]:
addon_service_columns = [
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
]

model_df["Num Addon Services"] = (
    model_df[addon_service_columns]
    .eq("Yes")
    .sum(axis=1)
)

model_df["Automatic Payment"] = (
    model_df["Payment Method"]
    .isin(["Bank transfer (automatic)", "Credit card (automatic)"])
    .astype(int)
)

model_df["Month-to-Month"] = (
    model_df["Contract"]
    .eq("Month-to-month")
    .astype(int)
)

model_df[[
    "Num Addon Services",
    "Automatic Payment",
    "Month-to-Month"
]].describe().T


,count,mean,std,min,25%,50%,75%,max
Num Addon Services,"7,043.00",2.04,1.85,0.00,0.00,2.00,3.00,6.00
Automatic Payment,"7,043.00",0.44,0.50,0.00,0.00,0.00,1.00,1.00
Month-to-Month,"7,043.00",0.55,0.50,0.00,0.00,1.00,1.00,1.00


## 5. Final predictor set

The target is separated from the predictors. No transformation is fitted before the train/test split.


In [7]:
X = model_df.drop(columns=[TARGET])
y = model_df[TARGET].copy()

print(f"Predictor columns: {X.shape[1]}")
print(f"Target rows: {len(y):,}")


Predictor columns: 22
Target rows: 7,043


## 6. Identify numeric and categorical features


In [8]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric features ({len(numeric_features)}):")
for col in numeric_features:
    print(f"- {col}")

print(f"\nCategorical features ({len(categorical_features)}):")
for col in categorical_features:
    print(f"- {col}")


Numeric features (6):
- Tenure Months
- Monthly Charges
- Total Charges
- Num Addon Services
- Automatic Payment
- Month-to-Month

Categorical features (16):
- Gender
- Senior Citizen
- Partner
- Dependents
- Phone Service
- Multiple Lines
- Internet Service
- Online Security
- Online Backup
- Device Protection
- Tech Support
- Streaming TV
- Streaming Movies
- Contract
- Paperless Billing
- Payment Method


## 7. Stratified train/test split

The target is imbalanced, so the split is stratified to preserve approximately the same churn proportion in both sets.

A fixed random seed ensures reproducibility.


In [9]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")

print(f"\nTraining churn rate: {100 * y_train.mean():.2f}%")
print(f"Test churn rate:     {100 * y_test.mean():.2f}%")


Training rows: 5,634
Test rows:     1,409

Training churn rate: 26.54%
Test churn rate:     26.54%


## 8. Build the preprocessing pipeline

### Numeric variables
- median imputation;
- standardization.

### Categorical variables
- most-frequent imputation;
- one-hot encoding;
- unseen categories ignored at prediction time.

The preprocessing pipeline is **fit only on the training data**.


In [12]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 9. Fit preprocessing on the training set only

This step is performed here to validate the feature pipeline and inspect the resulting transformed feature space.

In [14]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Original training shape:    {X_train.shape}")
print(f"Processed training shape:   {X_train_processed.shape}")
print(f"Original test shape:        {X_test.shape}")
print(f"Processed test shape:       {X_test_processed.shape}")


Original training shape:    (5634, 22)
Processed training shape:   (5634, 49)
Original test shape:        (1409, 22)
Processed test shape:       (1409, 49)


## 10. Inspect transformed feature names


In [15]:
feature_names = preprocessor.get_feature_names_out()

feature_name_df = pd.DataFrame({
    "processed_feature": feature_names
})

feature_name_df


,processed_feature
0,num__Tenure Months
1,num__Monthly Charges
2,num__Total Charges
3,num__Num Addon Services
4,num__Automatic Payment
5,num__Month-to-Month
6,cat__Gender_Female
7,cat__Gender_Male
8,cat__Senior Citizen_No
9,cat__Senior Citizen_Yes


## 11. Verify the processed data

The transformed matrices should contain no missing values.


In [16]:
def count_missing_in_matrix(matrix):
    if hasattr(matrix, "toarray"):
        return np.isnan(matrix.data).sum()
    return np.isnan(matrix).sum()

train_missing = count_missing_in_matrix(X_train_processed)
test_missing = count_missing_in_matrix(X_test_processed)

print(f"Missing values in processed training matrix: {train_missing}")
print(f"Missing values in processed test matrix:     {test_missing}")


Missing values in processed training matrix: 0
Missing values in processed test matrix:     0


## 12. Preserve customer value for the test set

The test-set CLTV values are saved separately so that predictions from the next notebook can later be combined with customer value for retention prioritization.

The indices are preserved before export so each test record can be matched correctly.


In [17]:
test_value = df.loc[X_test.index, ["CLTV", TARGET]].copy()
test_value["source_index"] = X_test.index

test_value.head()


,CLTV,Churn Value,source_index
2196,4842,0,2196
3549,5157,0,3549
3515,2894,0,3515
5162,2831,0,5162
4642,4324,0,4642


## 13. Export train/test datasets and metadata

Raw split datasets are exported rather than only the transformed matrices.

Outputs:

- `X_train.csv`
- `X_test.csv`
- `y_train.csv`
- `y_test.csv`
- `test_customer_value.csv`
- `model_feature_metadata.json`


In [18]:
if Path("../data").exists():
    OUTPUT_DIR = Path("../data/processed")
else:
    OUTPUT_DIR = Path("data/processed")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_export = X_train.copy()
X_train_export["source_index"] = X_train.index

X_test_export = X_test.copy()
X_test_export["source_index"] = X_test.index

y_train_export = pd.DataFrame({
    "source_index": y_train.index,
    TARGET: y_train.values,
})

y_test_export = pd.DataFrame({
    "source_index": y_test.index,
    TARGET: y_test.values,
})

X_train_export.to_csv(OUTPUT_DIR / "X_train.csv", index=False)
X_test_export.to_csv(OUTPUT_DIR / "X_test.csv", index=False)
y_train_export.to_csv(OUTPUT_DIR / "y_train.csv", index=False)
y_test_export.to_csv(OUTPUT_DIR / "y_test.csv", index=False)
test_value.to_csv(OUTPUT_DIR / "test_customer_value.csv", index=False)

metadata = {
    "target": TARGET,
    "excluded_from_prediction": ["CLTV"],
    "engineered_features": [
        "Num Addon Services",
        "Automatic Payment",
        "Month-to-Month",
    ],
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "processed_feature_count": len(feature_names),
}

with open(OUTPUT_DIR / "model_feature_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved feature-engineering outputs:")
for filename in [
    "X_train.csv",
    "X_test.csv",
    "y_train.csv",
    "y_test.csv",
    "test_customer_value.csv",
    "model_feature_metadata.json",
]:
    print(f"- {OUTPUT_DIR / filename}")


Saved feature-engineering outputs:
- ../data/processed/X_train.csv
- ../data/processed/X_test.csv
- ../data/processed/y_train.csv
- ../data/processed/y_test.csv
- ../data/processed/test_customer_value.csv
- ../data/processed/model_feature_metadata.json
